# Embedding Models

**Module:** 01 — Embeddings

A practical tour of widely used embedding model families and how to choose among them for cost, quality, language coverage, and deployment constraints.


## How to Use This Notebook

Read each section as a mini-lesson, run every code cell, then change inputs to stress-test your intuition. API examples use placeholders such as `YOUR_API_KEY` or `os.getenv(...)` — never hard-code secrets.

Each major topic includes: definition, why it matters, how it works, intuition, pitfalls, when-to-use guidance, practical demos, and a short exercise.


## Learning Objectives

By the end of this notebook, you will be able to:

- Compare major embedding model families at a practitioner level
- Map model choices to local vs API deployment
- Read model cards for dimension, metric, and instruction format
- Build a lightweight selection rubric for your project


## OpenAI — text-embedding-3-small / large

**Definition.** API embedding models with strong general quality and optional dimension shortening.

**Why it matters.** Fast path to solid retrieval without hosting GPUs; integrates with OpenAI ecosystem.

**How it works.** POST vectors via API; store dim; use cosine/IP on normalized vectors; version model name in index metadata.

**Intuition.** Rent a high-quality encoder instead of owning the factory.

**Common pitfalls.**
- Cost at very large re-embed volumes
- Data residency constraints

**When to use.** Hosted apps that already use OpenAI and allow sending text to the API.


In [ ]:
import os, json
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "YOUR_API_KEY")
request = {
    "model": "text-embedding-3-small",
    "input": "Refund window for unused items",
    "dimensions": 512,  # optional shortening
}
response = {
    "data": [{"embedding": [0.01, -0.02], "index": 0}],
    "usage": {"total_tokens": 8},
}
print(json.dumps({"request": request, "response": response}, indent=2))
print("auth header would be: Bearer", OPENAI_API_KEY[:6] + "...")


## BAAI — BGE (Small / Base / Large / M3)

**Definition.** Open embedding family popular for retrieval; M3 adds multilingual & multi-function capabilities.

**Why it matters.** Strong open weights for self-hosted search and RAG.

**How it works.** Run via Sentence Transformers / FlagEmbedding; check query instruction prefixes.

**Intuition.** Workhorse open models for many RAG tutorials and prod systems.

**Common pitfalls.**
- Using wrong prompt template
- Picking largest model without latency budget

**When to use.** Self-hosted semantic search, especially multilingual (M3).


In [ ]:
# Local-style usage pattern (weights not downloaded in this stub)
model_ids = {
    "small": "BAAI/bge-small-en-v1.5",
    "base": "BAAI/bge-base-en-v1.5",
    "large": "BAAI/bge-large-en-v1.5",
    "m3": "BAAI/bge-m3",
}
for k,v in model_ids.items():
    print(f"{k:5s} -> {v}")
print("Typical: normalize_embeddings=True, metric=cosine")


## E5 — small / base / large

**Definition.** Embedding models trained with weak supervision; require `query:` / `passage:` prefixes.

**Why it matters.** Excellent quality when instructions are applied correctly; common in open RAG stacks.

**How it works.** Prefix texts, encode, normalize, cosine search.

**Intuition.** A bilingual name tag: label the side of the conversation (query vs passage).

**Common pitfalls.**
- Forgetting prefixes silently tanks recall

**When to use.** Open retrieval systems willing to enforce formatting conventions.


In [ ]:
def e5_pair(query, passage):
    return "query: " + query, "passage: " + passage

print(e5_pair("warranty length?", "The warranty lasts two years."))


## Jina — jina-embeddings-v3

**Definition.** Modern embedding model family with strong retrieval focus and flexible task adapters / long context options (check current card).

**Why it matters.** Good candidate when you want open/commercial Jina ecosystem features.

**How it works.** Follow model card for task labels and recommended similarity metric.

**Intuition.** Task-aware embeddings — same backbone, different 'modes'.

**Common pitfalls.**
- Ignoring task-specific settings

**When to use.** Apps already standardized on Jina tooling or needing their feature set.


In [ ]:
# Request shape sketch for a hosted embeddings API
request = {
    "model": "jina-embeddings-v3",
    "input": ["short query", "long passage ..."],
    "task": "retrieval.query",  # illustrative
}
print(request)


## Nomic — nomic-embed-text

**Definition.** Open embedding model positioned for strong quality/context with open weights.

**Why it matters.** Attractive for self-hosted stacks seeking OpenAI-competitive open options.

**How it works.** Use with Sentence Transformers / GGUF deployments as documented; verify prefixes.

**Intuition.** Open alternative in the 'good default embedder' slot.

**Common pitfalls.**
- Not verifying sequence length limits for long chunks

**When to use.** Local/privacy-sensitive deployments.


In [ ]:
candidates = [
    ("nomic-embed-text", 768, "open"),
    ("text-embedding-3-small", 1536, "api"),
]
for name, dim, mode in candidates:
    print(f"{name:28s} dim={dim} mode={mode}")


## Sentence Transformers Library Models

**Definition.** A huge hub of community and official models loadable with one API (`SentenceTransformer`).

**Why it matters.** Fastest way to experiment locally across many checkpoints.

**How it works.** Pick model id → encode(list[str], normalize_embeddings=True) → cosine.

**Intuition.** A model zoo with a uniform steering wheel.

**Common pitfalls.**
- Randomly picking popular models without domain eval

**When to use.** Prototyping and many production self-hosted systems.


In [ ]:
# Pseudocode API (install sentence-transformers to run live)
pseudo = '''
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
emb = model.encode(["hello world"], normalize_embeddings=True)
'''
print(pseudo)


## Practical Selection Tips

**Definition.** Model selection is a multi-objective decision: quality on *your* eval set, latency, cost, language coverage, privacy, and ops complexity.

**Why it matters.** Leaderboards ≠ your tickets, policies, or jargon.

**How it works.** Build a 50–200 query golden set; measure Recall@k / nDCG; track p95 latency and $ per million tokens; document the winner.

**Intuition.** Bake-off with a fixed recipe book, not Instagram food photos.

**Common pitfalls.**
- Switching models weekly without re-embedding
- Optimizing only average quality while p95 latency burns UX

**When to use.** Before locking an index format for production.

| Criterion | Question to ask |
|---|---|
| Quality | Recall@10 on *our* paraphrases? |
| Languages | Do we need multilingual / cross-lingual? |
| Privacy | Can text leave our VPC? |
| Cost | Re-embed + query burn rate? |
| Latency | CPU/GPU budget for p95? |
| Ops | API SLA vs self-host expertise? |


In [ ]:
# Selection rubric scorer
options = [
    {"name": "emb-api-small", "quality": 4, "privacy": 2, "cost": 4, "ops": 5},
    {"name": "bge-base-local", "quality": 4, "privacy": 5, "cost": 3, "ops": 3},
    {"name": "minilm-local", "quality": 3, "privacy": 5, "cost": 5, "ops": 4},
]
weights = {"quality": 0.4, "privacy": 0.25, "cost": 0.2, "ops": 0.15}
for o in options:
    score = sum(o[k] * w for k, w in weights.items())
    print(f"{o['name']:16s} score={score:.2f}")


In [ ]:
# Offline bake-off harness sketch
from dataclasses import dataclass

@dataclass
class EvalQuery:
    query: str
    gold_ids: set

def recall_at_k(ranked_ids, gold, k=5):
    return len(set(ranked_ids[:k]) & gold) / (len(gold) + 1e-9)

evals = [EvalQuery("refund window", {"doc:policy3"})]
# pretend two models returned rankings
rankings = {
    "modelA": ["doc:shipping", "doc:policy3", "doc:password"],
    "modelB": ["doc:policy3", "doc:shipping", "doc:password"],
}
for name, ranked in rankings.items():
    scores = [recall_at_k(ranked, e.gold_ids, 5) for e in evals]
    print(name, "mean recall@5", round(sum(scores)/len(scores), 3))


### Try it yourself — Model bake-off

1. Write 10 paraphrase queries for one internal document.
2. Score two different embedding models with Recall@5.
3. Decide winner using the weighted rubric above (adjust weights for your constraints).


## Deep Dive — Operational Checklist

**Definition.** Production embedding systems are contracts: model ID, dimension, metric, prefixes, and preprocessing must match between index and query paths.

**Why it matters.** Silent contract drift is the most common cause of sudden recall collapse.

**How it works.** Store model metadata with the collection; fail closed on mismatch; re-embed on upgrades with a dual-read window if needed.

**Intuition.** Two maps with different projections cannot share GPS coordinates.

**Common pitfalls.**
- Swapping models without reindexing
- Comparing cosine thresholds across models
- Mixing instruction prefixes inconsistently

**When to use.** Every deployment—not just the first prototype.


In [ ]:
# Contract validator
contract = {
    "model": "text-embedding-3-small",
    "dim": 1536,
    "metric": "cosine",
    "normalize": True,
    "query_prefix": "",
    "doc_prefix": "",
}

def validate_vector(vec, contract):
    assert len(vec) == contract["dim"], (len(vec), contract["dim"])
    return True

validate_vector([0.0] * contract["dim"], contract)
print("contract ok", contract["model"])


### Try it yourself — Contract lab

1. Write the contract dict for your preferred open embedding model.
2. Intentionally break the dimension and show the assertion firing.
3. Document who owns re-embedding after a model upgrade.


In [ ]:
# Threshold calibration sketch
pairs = [("rel", 0.84), ("rel", 0.79), ("irr", 0.55), ("irr", 0.71)]
for thr in [0.70, 0.75, 0.80]:
    tp = sum(1 for y,s in pairs if y=="rel" and s>=thr)
    fp = sum(1 for y,s in pairs if y=="irr" and s>=thr)
    print(thr, "tp", tp, "fp", fp)


## Comparison table — practical choices

| Concern | Prefer | Avoid |
|---|---|---|
| Paraphrase FAQ | Dense / hybrid | Keywords alone |
| Invoice IDs | Lexical / hybrid | Dense-only |
| Privacy VPC | Local open model | Unapproved SaaS |
| Fast prototype | Small API/local MiniLM | Giant untested models |


In [ ]:
# Mini bake-off harness
rankings = {
    "modelA": ["d2", "d1", "d3"],
    "modelB": ["d1", "d2", "d3"],
}
gold = {"d1"}
for name, ranked in rankings.items():
    hit = ranked[0] in gold
    print(name, "top1_hit", hit)


```mermaid
flowchart LR
  A[Corpus] --> B[Embed+index]
  C[Query] --> D[Embed]
  D --> E[Search]
  B --> E
  E --> F[Evaluate recall]
  F -->|bad| G[Fix chunking/model/metric]
  F -->|good| H[Ship with monitors]
```


### Try it yourself — End-to-end

1. Build a 10-document toy index with hashed embeddings.
2. Create 5 paraphrase queries and compute Recall@3.
3. Write one monitoring alert you would page on in production.


In [ ]:
import numpy as np

def emb(text, dim=32):
    rng = np.random.default_rng(abs(hash(text.lower())) % (2**32))
    v = rng.normal(size=dim)
    return v / (np.linalg.norm(v) + 1e-9)

docs = {
    "d1": "Refunds are available for 14 days",
    "d2": "Reset your password via email",
    "d3": "Express shipping takes two days",
}
X = {i: emb(t) for i,t in docs.items()}
q = emb("how long can I return an item?")
ranked = sorted(((float(np.dot(q,v)), i) for i,v in X.items()), reverse=True)
print(ranked)
print("recall@1", float(ranked[0][1] == "d1"))


## Summary & Key Takeaways

- APIs optimize time-to-value; open models optimize control and privacy.
- Instruction prefixes and dimensions are part of the contract—store them with the index.
- Select with a domain eval set, not vibes or leaderboard screenshots alone.


## Self-Check

1. Can you explain the main idea of each section in one sentence?
2. Which technique would you use first in production, and why?
3. What failure mode should you monitor after shipping?
4. What metric would tell you the system got worse?


In [ ]:
checklist = [
    "I can restate the learning objectives",
    "I ran/adapted at least two code examples",
    "I know which env vars/keys this topic needs",
    "I noted one risk (cost, safety, latency, or quality)",
    "I can name one pitfall and its mitigation",
]
for i, item in enumerate(checklist, 1):
    print(f"{i}. [ ] {item}")
